# 29. Graph Neural Networks

Shengli (Bruce) Jiang, University of South Carolina

## Setup

TensorFlow 2.16.2 does not support Python 3.14. Create a separate Python 3.12 environment by running these commands in Terminal.

```bash
conda create -n eche589-gnn python=3.12 pip -y
conda activate eche589-gnn
python -m pip install setuptools==75.3.0 tensorflow==2.16.2 tf-keras==2.16.0 spektral==1.3.1 ipykernel
python -m ipykernel install --user --name eche589-gnn --display-name "ECHE 589 GNN"
```

In Jupyter, select **ECHE 589 GNN** as the notebook kernel.

In [1]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import numpy as np
import tensorflow as tf
import spektral

np.set_printoptions(precision=3, suppress=True)
print('TensorFlow', tf.__version__)
print('Spektral', spektral.__version__)

TensorFlow 2.16.2
Spektral 1.3.1


## Ethanol

Atom order: `C0-C1-O2`. The columns of `X` indicate carbon and oxygen; `A` records bonds.

In [2]:
X = np.array([
    [1, 0],
    [1, 0],
    [0, 1],
], dtype=np.float32)

A = np.array([
    [0, 1, 0],
    [1, 0, 1],
    [0, 1, 0],
], dtype=np.float32)

## GCN

`gcn_filter(A)` adds self-loops and normalizes the adjacency matrix:

$$S=\widetilde D^{-1/2}(A+I)\widetilde D^{-1/2},$$

with $\widetilde D_{ii}=\sum_j(A+I)_{ij}$.

In [3]:
from spektral.utils.convolution import gcn_filter

S = gcn_filter(A)
S

array([[0.5  , 0.408, 0.   ],
       [0.408, 0.333, 0.408],
       [0.   , 0.408, 0.5  ]], dtype=float32)

`channels=2` gives two output features per atom. The first call creates the layer weights.

In [4]:
from spektral.layers import GCNConv

gcn = GCNConv(channels=2, activation='relu', use_bias=True)
H = gcn([X, S])
H.shape

TensorShape([3, 2])

The same `W` and `b` apply to every atom.

In [5]:
W = np.array([[1, 0], [0, 2]], dtype=np.float32)
b = np.zeros(2, dtype=np.float32)

gcn.set_weights([W, b])
H = gcn([X, S])
H.numpy()

array([[0.908, 0.   ],
       [0.742, 0.816],
       [0.408, 1.   ]], dtype=float32)

`GCNConv` evaluates

$$H=\operatorname{ReLU}(SXW+b).$$

In [6]:
H_matrix = np.maximum(S @ X @ W + b, 0)
H_matrix

array([[0.908, 0.   ],
       [0.742, 0.816],
       [0.408, 1.   ]], dtype=float32)

## Readout

`GlobalSumPool` sums the rows of `H` into one molecular vector of shape `(1, 2)`.

In [7]:
from spektral.layers import GlobalSumPool

readout = GlobalSumPool()
g = readout(H)
g.numpy()

array([[2.058, 1.816]], dtype=float32)

## Graph Model

`Input(shape=...)` omits the first dimension. The full input shapes are `(N, 2)` and `(N, N)` for `N` atoms.

In [8]:
from tensorflow import keras

atom_input = keras.Input(shape=(2,), name='atom_features')
filter_input = keras.Input(shape=(None,), name='normalized_adjacency')

atom_states = gcn([atom_input, filter_input])
molecular_vector = readout(atom_states)
encoder = keras.Model([atom_input, filter_input], molecular_vector)

In [9]:
encoder([X, S]).numpy()

array([[2.058, 1.816]], dtype=float32)